# Chapter 8 — Metaclasses and Attributes

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 8.

---


### Item 58 — Use Plain Attributes Instead of Setter and Getter Methods

Coming from languages where `getX()`/`setX()` are the norm, it's tempting to write the same pattern in Python — but plain public attributes (`obj.x = 5`) are simpler, more Pythonic, and can be upgraded to a `@property` later (Item 59) without changing any calling code, since the attribute-access syntax stays identical either way.


In [1]:
# Getter/setter style: unnecessary ceremony for a value with no extra logic (yet)
class ResistorVerbose:
    def __init__(self, ohms):
        self._ohms = ohms
    def get_ohms(self):
        return self._ohms
    def set_ohms(self, value):
        self._ohms = value

r = ResistorVerbose(50)
r.set_ohms(100)
print(r.get_ohms())

# Plain attribute: does the exact same job, with normal, direct syntax
class Resistor:
    def __init__(self, ohms):
        self.ohms = ohms

r2 = Resistor(50)
r2.ohms = 100        # direct assignment -- no method call needed
r2.ohms += 25         # even in-place operators work naturally
print(r2.ohms)


100
125


### Item 59 — Consider `@property` Instead of Refactoring Attributes

If a plain attribute later needs validation or computed behavior, `@property` lets you add that logic *without breaking existing callers* — they keep writing `obj.x = 5` and `obj.x`, but those now run your getter/setter methods behind the scenes. This is exactly why Item 58 says start with plain attributes: you can always upgrade to `@property` later with zero changes to calling code.


In [2]:
class Resistor:
    def __init__(self, ohms):
        self._ohms = ohms

    @property
    def ohms(self):
        return self._ohms

    @ohms.setter
    def ohms(self, value):
        if value <= 0:
            raise ValueError('ohms must be positive')
        self._ohms = value

r = Resistor(50)
print(r.ohms)      # looks like plain attribute access, but actually calls the getter
r.ohms = 100        # looks like plain assignment, but actually calls the setter (with validation!)
print(r.ohms)

try:
    r.ohms = -5     # the validation added via @property kicks in transparently
except ValueError as e:
    print('ValueError:', e)


50
100
ValueError: ohms must be positive


### Item 60 — Use Descriptors for Reusable `@property` Methods

`@property` logic (like "validate this is a positive number") often needs repeating across several attributes or classes — copy-pasting the same getter/setter is exactly the duplication descriptors solve. A descriptor is a reusable class implementing `__get__`/`__set__` that you assign once as a class attribute; every attribute using that descriptor instance gets the same validation logic, written only once.


In [3]:
class PositiveNumber:
    """A reusable descriptor: enforces 'must be positive' for ANY attribute assigned to it."""
    def __set_name__(self, owner, name):
        self._name = '_' + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self._name, None)

    def __set__(self, instance, value):
        if value <= 0:
            raise ValueError(f'{self._name[1:]} must be positive')
        setattr(instance, self._name, value)

class Circle:
    radius = PositiveNumber()   # reused descriptor -- no per-class validation code needed

class Resistor:
    ohms = PositiveNumber()     # SAME descriptor class, reused on a totally different class

c = Circle()
c.radius = 5
print(c.radius)

r = Resistor()
r.ohms = 100
print(r.ohms)

try:
    c.radius = -1   # the exact same validation logic, defined only ONCE in PositiveNumber
except ValueError as e:
    print('ValueError:', e)


5
100
ValueError: radius must be positive


### Item 61 — Use `__getattr__`, `__getattribute__`, and `__setattr__` for Lazy Attributes

`__getattr__(self, name)` is called only when normal attribute lookup *fails* — perfect for lazily computing/loading an attribute the first time it's actually accessed (e.g., loading a value from a database on first use, then caching it). `__getattribute__` is called on *every* attribute access (rarely needed, powerful, easy to create infinite recursion with if you're not careful). `__setattr__` intercepts every assignment, useful for logging or validating writes generically.


In [4]:
class LazyRecord:
    def __init__(self, data_source):
        self._data_source = data_source
        # notice: no 'expensive_value' attribute exists yet at all

    def __getattr__(self, name):
        # only called because normal lookup for `name` FAILED -- i.e., first access
        print(f'(loading {name} lazily...)')
        value = self._data_source.get(name, f'no-such-field-{name}')
        setattr(self, name, value)   # cache it -- next access is a normal, fast lookup, no __getattr__ call
        return value

record = LazyRecord({'expensive_value': 42})
print(record.expensive_value)   # triggers __getattr__ -- 'loading' message prints
print(record.expensive_value)   # already cached now -- no 'loading' message this time

class LoggingSetattr:
    def __setattr__(self, name, value):
        print(f'setting {name} = {value!r}')
        super().__setattr__(name, value)   # still have to actually perform the assignment

obj = LoggingSetattr()
obj.x = 10   # every assignment is intercepted and logged


(loading expensive_value lazily...)
42
42
setting x = 10


### Item 62 — Validate Subclasses with `__init_subclass__`

`__init_subclass__` is called automatically on a base class the moment any subclass is *defined* (at class-creation time, not instantiation) — perfect for enforcing rules like "every subclass must define this attribute/method," catching mistakes the instant a bad subclass is written, rather than only when someone tries to use it.


In [5]:
class Plugin:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        if not hasattr(cls, 'name'):
            raise TypeError(f'{cls.__name__} must define a class attribute "name"')
        print(f'Plugin registered: {cls.name}')

class GoodPlugin(Plugin):
    name = 'good_plugin'   # satisfies the requirement -- validated at class-DEFINITION time

try:
    class BadPlugin(Plugin):
        pass   # missing 'name' -- caught immediately when this class statement runs, not later
except TypeError as e:
    print('TypeError:', e)


Plugin registered: good_plugin
TypeError: BadPlugin must define a class attribute "name"


### Item 63 — Register Class Existence with `__init_subclass__`

Beyond validation, `__init_subclass__` can automatically register every subclass into a lookup table the moment it's defined — a common pattern for plugin systems (build a `SERIALIZERS = {}` dict keyed by format name, and each new subclass adds itself with zero manual registration code elsewhere).


In [6]:
class Serializer:
    _registry = {}

    def __init_subclass__(cls, *, format_name, **kwargs):
        super().__init_subclass__(**kwargs)
        Serializer._registry[format_name] = cls   # self-registers automatically, no manual list to maintain

class JSONSerializer(Serializer, format_name='json'):
    def serialize(self, data):
        return f'{{...json for {data}...}}'

class XMLSerializer(Serializer, format_name='xml'):
    def serialize(self, data):
        return f'<data>{data}</data>'

print(Serializer._registry)   # both subclasses registered themselves automatically

serializer_cls = Serializer._registry['xml']
print(serializer_cls().serialize({'a': 1}))


{'json': <class '__main__.JSONSerializer'>, 'xml': <class '__main__.XMLSerializer'>}
<data>{'a': 1}</data>


### Item 64 — Annotate Class Attributes with `__set_name__`

When writing a descriptor (Item 60), it needs to know *which* attribute name it was assigned to, to store its private backing value without a collision between multiple descriptor instances on the same class. `__set_name__(self, owner, name)` is called automatically at class-creation time with exactly that information, so the descriptor doesn't need it passed in manually.


In [7]:
class NamedField:
    def __set_name__(self, owner, name):
        # called automatically when the CLASS is defined, telling us our own attribute name
        self.name = name
        self.private_name = '_' + name
        print(f'NamedField bound to attribute: {name}')

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.private_name, None)

    def __set__(self, instance, value):
        setattr(instance, self.private_name, value)

class Person:
    first_name = NamedField()   # __set_name__ fires here, telling this descriptor its name is 'first_name'
    last_name = NamedField()    # a SEPARATE descriptor instance, correctly knows it's 'last_name'

p = Person()
p.first_name = 'Ada'
p.last_name = 'Lovelace'
print(p.first_name, p.last_name)
print(p.__dict__)   # backing values correctly stored under distinct private names, no collision


NamedField bound to attribute: first_name
NamedField bound to attribute: last_name
Ada Lovelace
{'_first_name': 'Ada', '_last_name': 'Lovelace'}


### Item 65 — Consider Class Body Definition Order to Establish Relationships Between Attributes

Python executes a class body top-to-bottom like any other code block — attributes/descriptors defined earlier in the body can be referenced by ones defined later (e.g., a later field validating against an earlier one's value), but not the reverse. Being deliberate about this order lets you express dependencies between class attributes without extra machinery.


In [8]:
class Field:
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return f'Field({self.name!r})'

class Meta:
    fields_seen_so_far = []   # defined FIRST -- can be referenced/appended to by what follows

    a = Field('a')
    fields_seen_so_far.append(a.name)   # this line can see `fields_seen_so_far` because it came earlier

    b = Field('b')
    fields_seen_so_far.append(b.name)

print(Meta.fields_seen_so_far)   # order of definition determined what could reference what
print(Meta.a, Meta.b)


['a', 'b']
Field('a') Field('b')


### Item 66 — Prefer Class Decorators over Metaclasses for Composable Class Extensions

Metaclasses (`class Foo(metaclass=MyMeta)`) can customize class creation, but only **one** metaclass can apply to a given class, making them hard to compose — two independent metaclass-based features generally can't be combined on the same class. A class decorator (`@my_decorator` applied to the whole class, run *after* the class already exists) achieves similar customization, and — crucially — several decorators can be stacked freely on the same class.


In [9]:
def add_repr(cls):
    """Class decorator: adds a generic __repr__ based on __dict__."""
    def __repr__(self):
        fields = ', '.join(f'{k}={v!r}' for k, v in self.__dict__.items())
        return f'{cls.__name__}({fields})'
    cls.__repr__ = __repr__
    return cls

def add_equality(cls):
    """A SECOND, independent class decorator -- stacks freely, unlike two metaclasses would."""
    def __eq__(self, other):
        return isinstance(other, cls) and self.__dict__ == other.__dict__
    cls.__eq__ = __eq__
    return cls

@add_repr
@add_equality              # both decorators applied to the SAME class -- composes cleanly
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

p1, p2 = Point(1, 2), Point(1, 2)
print(p1)          # from add_repr
print(p1 == p2)     # from add_equality -- both features present together, no conflict


Point(x=1, y=2)
True
